# YZM 513 — Week 8 Warm-up: CUDA C/C++ Examples

Run the seven examples from **Part IV** of the slides. For each one: **predict first, run second, then change one thing.**

Select **Runtime → Change runtime type → T4 GPU** before you start.

## 0. Setup

Clone the course repository and check that the GPU and `nvcc` are visible.

In [ ]:
!nvidia-smi -L
!nvcc --version | tail -2

In [ ]:
!git clone -q https://github.com/adaskin/hpc-for-ai-course 2>/dev/null || true
%cd hpc-for-ai-course/lectures/week08/cuda_c
!ls

## Example 1: Hello, threads: the core formula

Before you run: write down the 12 values of `i` that you expect (blocks 0, 1, 2 × threads 0–3). Then run the cell. The *order* of the lines may differ from your list; the *set* of lines must not.

In [ ]:
!nvcc -o 01_hello_indices 01_hello_indices.cu && ./01_hello_indices

**Try it:** change the launch from `<<<3, 4>>>` to `<<<2, 5>>>` (the next cell does it with `sed`, runs the program, sorts the lines by `i`, and restores the file). Write down the ten values of `i` you expect *before* you run it.

In [ ]:
!sed -i 's/hello_indices<<<3, 4>>>/hello_indices<<<2, 5>>>/' 01_hello_indices.cu && nvcc -o 01_try 01_hello_indices.cu && ./01_try | sort -t= -k3 -n
!git checkout 01_hello_indices.cu

## Example 2: Vector add, N = 10: allocate, copy, launch, copy back, free

Before you run: fill in the table from the slides for threads 0–11 (`i`, `i < 10`?, `c[i]`). The program should print `c = 0 11 22 … 99`.

In [ ]:
!nvcc -o 02_vector_add_small 02_vector_add_small.cu && ./02_vector_add_small

**Try it:** change `N` to `13` and `threads` to `5`. How many blocks? How many idle threads? Then change `bytes` to `N` (a classic bug) and see what the output looks like.

In [ ]:
# Edit the .cu file (Colab file browser), then recompile and rerun:
!nvcc -o 02_vector_add_small 02_vector_add_small.cu && ./02_vector_add_small

## Example 3: The missing bounds check

Before you run: which two cells will the unsafe kernel overwrite, and with which values?

In [ ]:
!nvcc -o 03_bounds_check 03_bounds_check.cu && ./03_bounds_check

**Try it:** launch `<<<4, 4>>>` instead of `<<<3, 4>>>` in the *unsafe* run (16 threads for a 12-int buffer). Nothing visible may happen. Explain why that is *worse* than a crash, then run the program under `compute-sanitizer` to see the invalid writes reported.

In [ ]:
# Edit the .cu file (Colab file browser), then recompile and rerun:
!nvcc -o 03_bounds_check 03_bounds_check.cu && ./03_bounds_check
!compute-sanitizer ./03_bounds_check 2>&1 | head -20

## Example 4: Launch configuration

Before you run: compute `blocks`, launched threads and idle threads for `N = 1000` and `threads = 128` by hand. Check against Part C's logic.

In [ ]:
!nvcc -o 04_launch_config 04_launch_config.cu && ./04_launch_config

**Try it:** add `2048` to the `sizes` array in Part C. What error do you get, and which line reports it?

In [ ]:
# Edit the .cu file (Colab file browser), then recompile and rerun:
!nvcc -o 04_launch_config 04_launch_config.cu && ./04_launch_config

## Example 5: Error checking

Before you run: for each of the three failures, decide *which call* will report it. The `null-pointer launch` has **two** checks. Which one says OK?

In [ ]:
!nvcc -o 05_error_checking 05_error_checking.cu && ./05_error_checking

**Try it:** replace `write_one<<<1, 1>>>(nullptr)` by `write_one<<<1, 1>>>(d_ok)` and run again. Which lines change? Then run the program under `compute-sanitizer` (cell below).

In [ ]:
!compute-sanitizer ./05_error_checking 2>&1 | head -20

## Example 6: A 5 × 7 matrix: 2D indexing

Before you run: for block (1,0), thread (2,3), compute `col`, `row`, the flat index and the stored value. It should be 36.

In [ ]:
!nvcc -o 06_matrix_2d 06_matrix_2d.cu && ./06_matrix_2d

**Try it:** change the matrix to `n_rows = 6, n_cols = 9` and the block to `dim3 threads(8, 2)`. Predict the grid size and the number of idle threads.

In [ ]:
# Edit the .cu file (Colab file browser), then recompile and rerun:
!nvcc -o 06_matrix_2d 06_matrix_2d.cu && ./06_matrix_2d

## Example 7: Grid-stride loop

Before you run: with 8 threads and `N = 20`, which elements does thread 5 handle? How many elements does thread 2 handle?

In [ ]:
!nvcc -o 07_grid_stride 07_grid_stride.cu && ./07_grid_stride

**Try it:** launch `who_did_it<<<1, 5>>>` instead of `<<<2, 4>>>`. Predict the `thread :` row, then run it.

In [ ]:
# Edit the .cu file (Colab file browser), then recompile and rerun:
!nvcc -o 07_grid_stride 07_grid_stride.cu && ./07_grid_stride

## Reflection

Answer in one or two sentences each:

1. Why does `blockIdx.x * blockDim.x + threadIdx.x` give every thread a different number, with no gaps?
2. Why is `(N + threads - 1) / threads` correct, and what goes wrong with `N / threads`?
3. Which line in Example 5 shows that a kernel crash can be reported *after* the launch statement?